In [42]:
df.info()

NameError: name 'df' is not defined

In [9]:
df["gender"].value_counts()

NameError: name 'df' is not defined

In [10]:
df.isnull().sum()

NameError: name 'df' is not defined

In [ ]:
df["company"].nunique()

In [ ]:
df["name"].head(20)

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/users.csv")

df.head()

In [44]:
!ls /content/Voyage-Analytics/data/raw

flights.csv  hotels.csv  users.csv


In [45]:
import pandas as pd

# Load the datasets
users = pd.read_csv("/content/Voyage-Analytics/data/raw/users.csv")
hotels = pd.read_csv("/content/Voyage-Analytics/data/raw/hotels.csv")
flights = pd.read_csv("/content/Voyage-Analytics/data/raw/flights.csv")

# Check the first few rows of each dataset
print(users.head())
print(hotels.head())
print(flights.head())

   code company             name  gender  age
0     0    4You        Roy Braun    male   21
1     1    4You   Joseph Holsten    male   37
2     2    4You    Wilma Mcinnis  female   48
3     3    4You     Paula Daniel  female   23
4     4    4You  Patricia Carson  female   44
   travelCode  userCode     name               place  days   price    total  \
0           0         0  Hotel A  Florianopolis (SC)     4  313.02  1252.08   
1           2         0  Hotel K       Salvador (BH)     2  263.41   526.82   
2           7         0  Hotel K       Salvador (BH)     3  263.41   790.23   
3          11         0  Hotel K       Salvador (BH)     4  263.41  1053.64   
4          13         0  Hotel A  Florianopolis (SC)     1  313.02   313.02   

         date  
0  09/26/2019  
1  10/10/2019  
2  11/14/2019  
3  12/12/2019  
4  12/26/2019  
   travelCode  userCode                from                  to  flightType  \
0           0         0         Recife (PE)  Florianopolis (SC)  firstClas

In [46]:
# Basic Info about each dataset
print("Users Dataset Info:")
print(users.info())

print("Hotels Dataset Info:")
print(hotels.info())

print("Flights Dataset Info:")
print(flights.info())

# Check for missing values
print("Missing Values in Users Dataset:")
print(users.isnull().sum())

print("Missing Values in Hotels Dataset:")
print(hotels.isnull().sum())

print("Missing Values in Flights Dataset:")
print(flights.isnull().sum())

# Checking the first few records in each dataset
print("First 5 records in Users Dataset:")
print(users.head())

print("First 5 records in Hotels Dataset:")
print(hotels.head())

print("First 5 records in Flights Dataset:")
print(flights.head())

Users Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1340 entries, 0 to 1339
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   code     1340 non-null   int64 
 1   company  1340 non-null   object
 2   name     1340 non-null   object
 3   gender   1340 non-null   object
 4   age      1340 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 52.5+ KB
None
Hotels Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40552 entries, 0 to 40551
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   travelCode  40552 non-null  int64  
 1   userCode    40552 non-null  int64  
 2   name        40552 non-null  object 
 3   place       40552 non-null  object 
 4   days        40552 non-null  int64  
 5   price       40552 non-null  float64
 6   total       40552 non-null  float64
 7   date        40552 non-null  object 
dtypes: float64(2), 

In [47]:
import pandas as pd

# Create a sample ratings DataFrame, assuming user interactions with hotels
ratings = pd.DataFrame({
    'user_id': [1, 1, 2, 2, 3, 3],    # User IDs
    'hotel_id': [101, 102, 101, 103, 102, 104],  # Hotel IDs
    'rating': [5, 4, 3, 5, 2, 4]    # Ratings (or interactions)
})

# Display the ratings DataFrame
print(ratings)

   user_id  hotel_id  rating
0        1       101       5
1        1       102       4
2        2       101       3
3        2       103       5
4        3       102       2
5        3       104       4


In [48]:
# Create the user-item matrix (ratings matrix)
user_item_matrix = pd.pivot_table(ratings, values='rating', index='user_id', columns='hotel_id', fill_value=0)

# Display the user-item matrix
print(user_item_matrix)

hotel_id  101  102  103  104
user_id                     
1         5.0  4.0  0.0  0.0
2         3.0  0.0  5.0  0.0
3         0.0  2.0  0.0  4.0


In [49]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Calculate cosine similarity between users
cosine_sim = cosine_similarity(user_item_matrix)

# Convert to DataFrame for better readability
cosine_sim_df = pd.DataFrame(cosine_sim, index=user_item_matrix.index, columns=user_item_matrix.index)

# Display the cosine similarity matrix
print(cosine_sim_df)

user_id         1         2         3
user_id                              
1        1.000000  0.401754  0.279372
2        0.401754  1.000000  0.000000
3        0.279372  0.000000  1.000000


In [50]:
def recommend_hotels(user_id, cosine_sim_df, user_item_matrix, top_n=5):
    # Get the most similar users to the given user
    similar_users = cosine_sim_df[user_id].sort_values(ascending=False)[1:]  # Exclude the user itself

    # Get the top N similar users
    top_users = similar_users.head(top_n).index

    # Get the hotels that these similar users interacted with
    recommended_hotels = user_item_matrix.loc[top_users].sum(axis=0)

    # Sort the hotels by the total interaction score and return the top N
    recommended_hotels = recommended_hotels.sort_values(ascending=False).head(top_n)

    return recommended_hotels

# Example usage for a user
user_id = 1  # Replace with the actual user_id
recommended_hotels = recommend_hotels(user_id, cosine_sim_df, user_item_matrix)
print(f"Recommended Hotels for User {user_id}:")
print(recommended_hotels)

Recommended Hotels for User 1:
hotel_id
103    5.0
104    4.0
101    3.0
102    2.0
dtype: float64


In [51]:
from sklearn.metrics import mean_squared_error
import numpy as np

# Assuming you have true ratings and predicted ratings
# Example: predicted_ratings and actual_ratings
predicted_ratings = [4.5, 3.0, 5.0, 4.0]  # replace with actual predictions
actual_ratings = [5.0, 3.0, 4.0, 4.0]  # replace with actual ratings

# Compute RMSE
rmse = np.sqrt(mean_squared_error(actual_ratings, predicted_ratings))
print(f"RMSE: {rmse}")

RMSE: 0.5590169943749475


In [52]:
!git add eda_users.ipynb

In [53]:
!git config --global user.email "pjindal2@calstatela.edu"
!git config --global user.name "pjindal01"

In [62]:
!git rm --cached notebooks/Voyage-Analytics

fatal: pathspec 'notebooks/Voyage-Analytics' did not match any files


In [61]:
!git add .

hint: You've added another git repository inside your current repository.
hint: Clones of the outer repository will not contain the contents of
hint: the embedded repository and will not know how to obtain it.
hint: If you meant to add a submodule, use:
hint: 
hint: 	git submodule add <url> notebooks/Voyage-Analytics
hint: 
hint: If you added this path by mistake, you can remove it from the
hint: index with:
hint: 
hint: 	git rm --cached notebooks/Voyage-Analytics
hint: 
hint: See "git help submodule" for more information.


In [54]:
!git commit -m "Updated EDA and recommendation model steps in eda_users notebook"

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	Voyage-Analytics/

nothing added to commit but untracked files present (use "git add" to track)


In [55]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address


In [57]:
!pwd

/content/Voyage-Analytics/notebooks


In [58]:
!git clone https://github.com/Nandishwar04/Voyage-Analytics.git

fatal: destination path 'Voyage-Analytics' already exists and is not an empty directory.


In [59]:
%cd /content/Voyage-Analytics/notebooks

/content/Voyage-Analytics/notebooks


In [60]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	Voyage-Analytics/

nothing added to commit but untracked files present (use "git add" to track)


In [63]:
!git ls-files

Voyage-Analytics
eda_users.ipynb


In [64]:
!git ls-files

Voyage-Analytics
eda_users.ipynb


In [66]:
!git rm -rf --cached notebooks/Voyage-Analytics

fatal: pathspec 'notebooks/Voyage-Analytics' did not match any files


In [67]:
!ls notebooks/

ls: cannot access 'notebooks/': No such file or directory


In [68]:
!ls

eda_users.ipynb  Voyage-Analytics
